# NB6 - ICDAR 2015 RT-DETR-l baseline
The Transformer arm NB3 left out (`RUN_RTDETR=False`). Trains RT-DETR-l on ICDAR 2015 for 100 epochs, evaluates on the held-out test split, and pushes the weights with a read-back check. Self-contained: mounts `icdar2015-task41-raw` and rebuilds ICDAR itself, exactly as NB3 does. Does not depend on NB0.

## 0. One-time environment setup

In [ ]:
# Same pins as the other notebooks. Uninstall ray to avoid the broken raytune callback.
! pip install -q ultralytics==8.3.40 pillow==10.4.0
! pip uninstall -y -q ray 2>/dev/null || true

## 1. Imports, constants, seeds

In [ ]:
# Standard library
import json, os, sys, shutil, subprocess, time, random, zipfile
from pathlib import Path

# Block ultralytics 8.3.40's broken raytune callback
sys.modules['ray'] = None

# Numeric and plotting
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

# Deep learning
import torch

# Reproducibility
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Paths (Kaggle)
KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT   = Path('/kaggle/input')
DATA_ROOT      = KAGGLE_WORKING / 'data'
ICDAR_DIR      = DATA_ROOT / 'icdar2015'
RUNS_DIR       = KAGGLE_WORKING / 'runs'
RESULTS_DIR    = KAGGLE_WORKING / 'results'
STATE_FILE     = KAGGLE_WORKING / 'nb6_state.json'

# Training config. Epochs match the YOLOv8s / YOLO-World ICDAR runs in NB3 (100),
# so the RT-DETR row is a like-for-like comparison, not the 50-epoch plan sacrifice.
IMGSZ         = 640
EPOCHS_RTDETR = 100
BATCH_RTDETR  = 8            # avoids OOM on T4
OPTIMIZER     = 'auto'
LR0, LRF      = 0.01, 0.01
PATIENCE      = 50
DEVICE        = 0
RTDETR_WEIGHTS = 'rtdetr-l.pt'

DATASET_SLUG = 'ausdm-v3-icdar-rtdetr'
KAGGLE_USERNAME_OVERRIDE = ''   # set if auto-detect returns 'unknown'

for d in [DATA_ROOT, ICDAR_DIR, RUNS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'KAGGLE_WORKING = {KAGGLE_WORKING}')
print(f'EPOCHS_RTDETR  = {EPOCHS_RTDETR}  BATCH = {BATCH_RTDETR}')
print(f'DATASET_SLUG   = {DATASET_SLUG}')
print(f'SEED           = {SEED}')

## 2. Resumable state and restore

In [ ]:
# _load_state / _mark_done / _is_done - stage-level idempotency across sessions
def _load_state():
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text())
    return {}

def _mark_done(stage, extra=None):
    st = _load_state()
    st[stage] = {'done': True, 'ts': time.time()}
    if extra:
        st[stage].update(extra)
    STATE_FILE.write_text(json.dumps(st, indent=2))

def _is_done(stage):
    return _load_state().get(stage, {}).get('done', False)

# _restore_from_prior_push - pull a prior NB6 output back into working
def _restore_from_prior_push():
    if STATE_FILE.exists():
        return 'local state already present'
    if not KAGGLE_INPUT.exists():
        return 'no /kaggle/input'
    for cand in KAGGLE_INPUT.rglob('nb6_state.json'):
        src = cand.parent
        shutil.copy2(cand, STATE_FILE)
        for f in ['icdar_rtdetr_metrics.csv']:
            if (src / f).exists():
                shutil.copy2(src / f, RESULTS_DIR / f)
        for sub in ['weights', 'figures']:
            if (src / sub).exists():
                shutil.copytree(src / sub, RESULTS_DIR / sub, dirs_exist_ok=True)
        return f'restored from {src}'
    return 'no prior push mounted'

print('restore:', _restore_from_prior_push())
print('current state:', json.dumps(_load_state(), indent=2))

## 3. Kaggle username and ICDAR mount - fail before any compute

In [ ]:
# _kaggle_user - try every credential source Kaggle might expose, in order
def _kaggle_user():
    if KAGGLE_USERNAME_OVERRIDE:
        return KAGGLE_USERNAME_OVERRIDE
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
        u = api.config_values.get('username') or api.get_config_value('username')
        if u:
            return u
    except Exception:
        pass
    u = os.environ.get('KAGGLE_USERNAME')
    if u:
        return u
    kj = Path.home() / '.kaggle' / 'kaggle.json'
    if kj.exists():
        try:
            return json.loads(kj.read_text())['username']
        except Exception:
            pass
    if KAGGLE_INPUT.exists():
        for p in KAGGLE_INPUT.rglob('*'):
            parts = p.parts
            if 'datasets' in parts:
                i = parts.index('datasets')
                if i + 1 < len(parts):
                    return parts[i + 1]
    return 'unknown'

KAGGLE_USER = _kaggle_user()
print('Kaggle username:', KAGGLE_USER)

# _find_icdar_mount - locate icdar2015-task41-raw wherever Kaggle placed it
def _find_icdar_mount():
    if not KAGGLE_INPUT.exists():
        return None
    def _has_marker(d):
        return ((d / 'ch4_training_images').exists() or
                (d / 'ch4_training_images.zip').exists())
    exact = KAGGLE_INPUT / 'icdar2015-task41-raw'
    if exact.exists() and _has_marker(exact):
        return exact
    for root, dirs, _files in os.walk(KAGGLE_INPUT):
        depth = Path(root).relative_to(KAGGLE_INPUT).parts
        if len(depth) > 3:
            dirs[:] = []; continue
        if _has_marker(Path(root)):
            return Path(root)
    return None

ICDAR_MOUNT = _find_icdar_mount()
assert ICDAR_MOUNT is not None, (
    'ICDAR raw dataset not mounted. Add Data -> "icdar2015-task41-raw" -> Add, '
    f'then re-run. /kaggle/input currently: '
    f'{[p.name for p in KAGGLE_INPUT.iterdir()] if KAGGLE_INPUT.exists() else "nothing"}')
print('ICDAR mount   :', ICDAR_MOUNT)
if KAGGLE_USER == 'unknown':
    print('WARNING: set KAGGLE_USERNAME_OVERRIDE in section 1 before the dry-run push.')

## 4. Rebuild ICDAR 2015 - unpack and polygon-to-axis-aligned conversion
Identical conversion to NB3, so the test split RT-DETR is scored on matches every other ICDAR number in the paper.

In [ ]:
# _unpack_icdar - handle either raw zips or Kaggle-auto-extracted folders
TEST_GT_CAND_ZIP = ['Challenge4_Test_Task1_GT.zip','Challenge4_Test_Task4_GT.zip',
                    'ch4_test_localization_transcription_gt.zip','ch4_test_gt.zip']
TEST_GT_CAND_DIR = ['Challenge4_Test_Task1_GT','Challenge4_Test_Task4_GT',
                    'ch4_test_localization_transcription_gt','ch4_test_gt']

def _unpack_icdar():
    zips = {'train_images':'ch4_training_images.zip',
            'train_gt':'ch4_training_localization_transcription_gt.zip',
            'test_images':'ch4_test_images.zip', 'test_gt':None}
    for cand in TEST_GT_CAND_ZIP:
        if (ICDAR_MOUNT / cand).exists():
            zips['test_gt'] = cand; break
    any_zip = any((ICDAR_MOUNT / v).exists() for v in zips.values() if v)
    if not any_zip:
        dirs = {'train_images':'ch4_training_images',
                'train_gt':'ch4_training_localization_transcription_gt',
                'test_images':'ch4_test_images', 'test_gt':None}
        for cand in TEST_GT_CAND_DIR:
            if (ICDAR_MOUNT / cand).is_dir():
                dirs['test_gt'] = cand; break
        assert dirs['test_gt'], 'No test GT folder or zip found in mount'
        for k, d in dirs.items():
            assert (ICDAR_MOUNT / d).is_dir(), f'Missing folder: {d}'
        return ICDAR_MOUNT, dirs
    assert zips['test_gt'], 'Test GT zip not found - check upload naming'
    stage_dir = KAGGLE_WORKING / 'icdar_raw'; stage_dir.mkdir(exist_ok=True)
    for key, name in zips.items():
        out = stage_dir / key
        if out.exists() and any(out.iterdir()):
            continue
        out.mkdir(exist_ok=True)
        with zipfile.ZipFile(ICDAR_MOUNT / name) as z:
            z.extractall(out)
    return stage_dir, {k: k for k in zips}

# _find_dir_with - locate the actual images/gt root inside an archive
def _find_dir_with(root, glob_pat):
    if any(root.glob(glob_pat)):
        return root
    for p in root.rglob('*'):
        if p.is_dir() and any(p.glob(glob_pat)):
            return p
    return root

# _convert_pair - polygons -> axis-aligned YOLO labels; drop '###' transcriptions
def _convert_pair(img_dir, gt_dir, out_split):
    (ICDAR_DIR / out_split / 'images').mkdir(parents=True, exist_ok=True)
    (ICDAR_DIR / out_split / 'labels').mkdir(parents=True, exist_ok=True)
    n_img = n_box = n_drop = 0
    for img in sorted(img_dir.iterdir()):
        if img.suffix.lower() not in {'.jpg','.jpeg','.png'}: continue
        gt = gt_dir / f'gt_{img.stem}.txt'
        if not gt.exists(): continue
        text = gt.read_text(encoding='utf-8-sig', errors='ignore')
        im = Image.open(img); W, H = im.size
        lines_out = []
        for ln in text.splitlines():
            if not ln.strip(): continue
            parts = ln.split(',', 8)
            if len(parts) < 8: continue
            try:
                xs = [float(parts[i]) for i in (0,2,4,6)]
                ys = [float(parts[i]) for i in (1,3,5,7)]
            except ValueError: continue
            trans = parts[8].strip() if len(parts) >= 9 else ''
            if trans == '###':
                n_drop += 1; continue
            x_min, x_max = max(0, min(xs)), min(W, max(xs))
            y_min, y_max = max(0, min(ys)), min(H, max(ys))
            bw = x_max - x_min; bh = y_max - y_min
            if bw <= 1 or bh <= 1: continue
            cx = (x_min + x_max)/2/W; cy = (y_min + y_max)/2/H
            lines_out.append(f'0 {cx:.6f} {cy:.6f} {bw/W:.6f} {bh/H:.6f}')
            n_box += 1
        (ICDAR_DIR / out_split / 'labels' / f'{img.stem}.txt').write_text(
            '\n'.join(lines_out) + ('\n' if lines_out else ''))
        shutil.copy2(img, ICDAR_DIR / out_split / 'images' / img.name)
        n_img += 1
    return n_img, n_box, n_drop

if not _is_done('icdar_convert'):
    ICDAR_RAW, SUB = _unpack_icdar()
    tr_img = _find_dir_with(ICDAR_RAW / SUB['train_images'], '*.jpg')
    tr_gt  = _find_dir_with(ICDAR_RAW / SUB['train_gt'],     'gt_*.txt')
    te_img = _find_dir_with(ICDAR_RAW / SUB['test_images'],  '*.jpg')
    te_gt  = _find_dir_with(ICDAR_RAW / SUB['test_gt'],      'gt_*.txt')
    tr = _convert_pair(tr_img, tr_gt, 'train')
    te = _convert_pair(te_img, te_gt, 'test')
    print(f'train: {tr[0]} imgs, {tr[1]} boxes, {tr[2]} dropped')
    print(f'test : {te[0]} imgs, {te[1]} boxes, {te[2]} dropped')
    assert tr[0] >= 900 and te[0] >= 450, f'Unexpected counts: train {tr[0]}, test {te[0]}'
    _mark_done('icdar_convert', {'train_imgs': tr[0], 'test_imgs': te[0],
                                 'train_boxes': tr[1], 'test_boxes': te[1]})
else:
    print('SKIP icdar_convert:', _load_state()['icdar_convert'])

ICDAR_YAML = ICDAR_DIR / 'data.yaml'
ICDAR_YAML.write_text(f'''path: {ICDAR_DIR}
train: train/images
val:   test/images
test:  test/images
nc: 1
names: ['text']
''')
print('icdar data.yaml:', ICDAR_YAML,
      '| test images:', len(list((ICDAR_DIR/'test'/'images').iterdir())))

## 5. Neutralise the raytune callback

In [ ]:
try:
    from ultralytics.utils.callbacks import raytune as _rt
    _rt.on_fit_epoch_end.__code__ = (lambda trainer: None).__code__
    _rt.callbacks = {}
    _rt.tune = None
    print('raytune callback neutralised (code swapped to no-op)')
except Exception as e:
    print('raytune patch skipped:', e)

## 6. Authenticated dry-run push - surface auth failures before training

In [ ]:
PUSH_DIR = KAGGLE_WORKING / 'push_stage'

# _stage_push_dir - rebuild staging folder and metadata
def _stage_push_dir(files=None, dirs=None):
    if PUSH_DIR.exists():
        shutil.rmtree(PUSH_DIR)
    PUSH_DIR.mkdir(parents=True, exist_ok=True)
    for f in (files or []):
        if Path(f).exists():
            shutil.copy2(f, PUSH_DIR / Path(f).name)
    for d in (dirs or []):
        if Path(d).exists():
            shutil.copytree(d, PUSH_DIR / Path(d).name, dirs_exist_ok=True)
    meta = {'title': 'AusDM v3 ICDAR RT-DETR',
            'id': f'{KAGGLE_USER}/{DATASET_SLUG}',
            'licenses': [{'name': 'CC-BY-SA-4.0'}]}
    (PUSH_DIR / 'dataset-metadata.json').write_text(json.dumps(meta, indent=2))
    return meta

# _push - create on first run, version-bump thereafter
def _push(message):
    # Robust push. The kaggle CLI can exit 0 while printing 'already in use',
    # so success is judged from the output text, not the return code. If the
    # dataset already exists, version-bump instead of create.
    def _run(sub):
        r = subprocess.run(['kaggle','datasets'] + sub, capture_output=True, text=True)
        return (r.stdout + r.stderr)
    out = _run(['create','-p',str(PUSH_DIR),'-r','zip'])
    if 'invalid owner id' in out.lower():
        print('CREATE:', out.strip()[-200:]); return False
    if 'is being created' in out.lower() or 'successfully' in out.lower():
        print('CREATE ok'); return True
    vout = _run(['version','-p',str(PUSH_DIR),'-m',message,'-r','zip'])
    print('VERSION:', vout.strip()[-220:])
    return ('is being created' in vout.lower()) or ('successfully' in vout.lower())
assert KAGGLE_USER != 'unknown', 'Set KAGGLE_USERNAME_OVERRIDE in section 1 and re-run.'

(RESULTS_DIR / 'nb6_marker.txt').write_text('nb6 dry-run ok\n')
_stage_push_dir(files=[RESULTS_DIR / 'nb6_marker.txt'])
if _push('NB6 dry-run'):
    print('dry-run push OK ->', f'{KAGGLE_USER}/{DATASET_SLUG}')
else:
    raise RuntimeError('Dry-run push failed. Fix auth / username before training.')

## 7. Train RT-DETR-l (100 epochs, batch 8)

In [ ]:
RTDETR_RUN  = RUNS_DIR / 'rtdetr_icdar'
RTDETR_BEST = RTDETR_RUN / 'weights' / 'best.pt'

# Train RT-DETR-l for 100 epochs, batch 8, same optimizer/lr/seed as the other ICDAR models.
if not _is_done('train_rtdetr') and not RTDETR_BEST.exists():
    from ultralytics import RTDETR
    model = RTDETR(RTDETR_WEIGHTS)
    model.train(data=str(ICDAR_YAML), epochs=EPOCHS_RTDETR, imgsz=IMGSZ, batch=BATCH_RTDETR,
                optimizer=OPTIMIZER, lr0=LR0, lrf=LRF, patience=PATIENCE, seed=SEED,
                device=DEVICE, project=str(RUNS_DIR), name='rtdetr_icdar',
                exist_ok=True, verbose=False, plots=True)
    assert RTDETR_BEST.exists(), 'RT-DETR-l best.pt not written'
    _mark_done('train_rtdetr', {'best': str(RTDETR_BEST)})
    print('RT-DETR-l training done ->', RTDETR_BEST)
else:
    print('SKIP train_rtdetr, best at', RTDETR_BEST)

## 8. Evaluate on the test split, checkpoint, and push

In [ ]:
# _eval_test - test-split metrics for an RT-DETR checkpoint
def _eval_test(weights, name):
    from ultralytics import RTDETR
    m = RTDETR(str(weights))
    mt = m.val(data=str(ICDAR_YAML), split='test', imgsz=IMGSZ, device=DEVICE,
               project=str(RUNS_DIR), name=name, exist_ok=True, verbose=False, plots=False)
    return {'mAP50-95': round(float(mt.box.map),4), 'mAP50': round(float(mt.box.map50),4),
            'precision': round(float(mt.box.mp),4), 'recall': round(float(mt.box.mr),4)}

rtdetr_test = _eval_test(RTDETR_BEST, 'rtdetr_test')
rtdetr_test.update({'model':'rtdetr-l', 'split':'test'})
print('RT-DETR-l test metrics:', json.dumps(rtdetr_test, indent=2))
_mark_done('eval_rtdetr', {'metrics': rtdetr_test})

# Single-row CSV, same columns as icdar_metrics.csv, ready to append into Table 1 in NB5
df_rtdetr = pd.DataFrame([rtdetr_test])[['model','split','mAP50-95','mAP50','precision','recall']]
df_rtdetr.to_csv(RESULTS_DIR / 'icdar_rtdetr_metrics.csv', index=False)
print(df_rtdetr.to_string(index=False))

# Checkpoint the weights immediately. These are the real payload that was lost last time.
(RESULTS_DIR / 'weights').mkdir(exist_ok=True)
shutil.copy2(RTDETR_BEST, RESULTS_DIR / 'weights' / 'rtdetr-l_icdar_best.pt')
rc = RTDETR_RUN / 'results.csv'
if rc.exists(): shutil.copy2(rc, RESULTS_DIR / 'rtdetr-l_results.csv')
shutil.copy2(STATE_FILE, RESULTS_DIR / 'nb6_state.json')
_stage_push_dir(dirs=[RESULTS_DIR])
print('RT-DETR checkpoint push:', 'OK' if _push('NB6 RT-DETR checkpoint') else 'FAILED')

**8.1 Explanation.**
- This is the missing Transformer row for ICDAR. Compare against the NB3 numbers already in `icdar_metrics.csv`: YOLOv8s 0.4352 and fine-tuned YOLO-World 0.4214 mAP50-95.
- A value in the same ~0.4 band confirms the dataset-difficulty story holds across paradigms; a much lower value would say RT-DETR struggles on dense small scene text. Read the printed `mAP50-95`.
- The weights are copied into `results/weights/` and pushed straight away, before anything else can interrupt the session.

## 9. Prediction sanity figures - boxes on test images

In [ ]:
# _draw - draw RT-DETR predicted boxes on n test images; count detections
FIG_DIR = RESULTS_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)

def _draw(weights, name, n=6, conf=0.25):
    from ultralytics import RTDETR
    m = RTDETR(str(weights))
    imgs = sorted((ICDAR_DIR / 'test' / 'images').iterdir())[:n]
    ndet = 0
    for img_path in imgs:
        res = m.predict(str(img_path), imgsz=IMGSZ, conf=conf, device=DEVICE, verbose=False)[0]
        im = Image.open(img_path).convert('RGB'); dr = ImageDraw.Draw(im)
        for b in res.boxes:
            x1,y1,x2,y2 = [float(v) for v in b.xyxy[0]]
            dr.rectangle([x1,y1,x2,y2], outline='lime', width=3); ndet += 1
        im.save(FIG_DIR / f'{name}_{img_path.stem}.png')
    return len(imgs), ndet

n1, d1 = _draw(RTDETR_BEST, 'pred_rtdetr_icdar')
print(f'RT-DETR-l: {n1} imgs, {d1} boxes drawn')

In [ ]:
# Inline preview so the box check happens inside the notebook, not just on disk
from IPython.display import Image as IPImg, display
for p in sorted(FIG_DIR.glob('pred_rtdetr_icdar_*'))[:4]:
    print(p.name); display(IPImg(str(p), width=460))

**9.1 Explanation.** Green boxes should cluster on scene-text regions. This is the qualitative evidence behind the RT-DETR row and must be visually confirmed here, since a boxless prediction figure is the exact defect Reviewer #2 caught in v2.

## 10. Final push with read-back verification

In [ ]:
# _verify_remote - read the pushed dataset back and confirm the payload really landed.
# This is the check that was missing when the weights were silently lost: judge success
# from the effect (the remote file list), never from the push command's exit code.
def _verify_remote(expect=('rtdetr-l_icdar_best.pt', 'icdar_rtdetr_metrics.csv')):
    ref = f'{KAGGLE_USER}/{DATASET_SLUG}'
    r = subprocess.run(['kaggle','datasets','files','-v', ref], capture_output=True, text=True)
    out = (r.stdout + r.stderr)
    missing = [f for f in expect if f not in out]
    print(out.strip()[-500:])
    print('missing from remote:', missing if missing else 'none')
    return not missing

_mark_done('final_push')
shutil.copy2(STATE_FILE, RESULTS_DIR / 'nb6_state.json')
_stage_push_dir(dirs=[RESULTS_DIR])
print('final push:', 'OK' if _push('NB6 final: ICDAR RT-DETR-l') else 'FAILED')

# Kaggle processes a new version asynchronously, so retry the read-back a few times.
verified = False
for attempt in range(4):
    time.sleep(30)
    if _verify_remote():
        verified = True; break
    print(f'  not confirmed yet (attempt {attempt+1}/4), waiting...')
print('READ-BACK VERIFY:', 'PASS' if verified else 'FAIL')
if not verified:
    print('WARNING: push not confirmed on the remote dataset. Download the notebook Output')
    print('         (/kaggle/working/results, incl. weights/rtdetr-l_icdar_best.pt) manually NOW.')
    print('         Do not close the session until the weights are saved somewhere off Kaggle.')

## 11. Conclusion

In [ ]:
print('=== NB6 summary ===')
print(pd.read_csv(RESULTS_DIR / 'icdar_rtdetr_metrics.csv').to_string(index=False))
print()
print('GPU         :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Epochs      :', EPOCHS_RTDETR, '(matches YOLOv8s / YOLO-World on ICDAR, like-for-like)')
print('Pushed      :', f'{KAGGLE_USER}/{DATASET_SLUG}')
print('Backup      : download the notebook Output (/kaggle/working/results) once, in case the push is lost')
print('Next in NB5 : append this row to icdar_metrics.csv and Table 1 (rtdetr-l, ICDAR, supervised)')